# Custom ICP Experiments
Experimenting with PyTorch3D's ICP implementation. We will use this baseline to measure registration against Chamfer Distance and visualize the resulting alignments.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import torch
import open3d as o3d
import numpy as np
import copy
import warnings
from typing import List, NamedTuple, Optional, TYPE_CHECKING, Union
from tqdm.notebook import tqdm

from chamferdist import ChamferDistance

# PyTorch3D utilities needed for ICP
from pytorch3d.ops import knn_points
from pytorch3d.structures import utils as strutil
from pytorch3d.ops import utils as oputil

# aside from matching by shortest distance, also match by density around each point
# density for each point is measured by the sum of its distances to its k neighbours in the same cloud
def calc_balanced_chamfer_loss_tensor(x, y, k=32, return_assignment=False, return_dists=False):
    chamferDist = ChamferDistance()
    eps = 0.00001
    k2 = k # reduce k to check density in smaller patches
    power = 4
    
    # add a loss term for mismatched pairs
    nn = chamferDist(x, y, bidirectional=True, return_nn=True, k=k)

    # measure density with itself
    nn_x = chamferDist(x, x, bidirectional=False, return_nn=True, k=k2)
    density_x = torch.mean(nn_x[0].dists[:,:,1:], dim=2)
    density_x = 1 / (density_x + eps)
    high, low = torch.max(density_x), torch.min(density_x)
    diff = high - low + eps
    density_x = (density_x - low) / diff

    # measure density with other cloud
    density_xy = torch.mean(nn[0].dists[:,:,:k2-1], dim=2)
    density_xy = 1 / (density_xy + eps)
    high, low = torch.max(density_xy), torch.min(density_xy)
    diff = high - low + eps
    density_xy = (density_xy - low) / diff
    w_x = torch.div(density_xy, density_x)
    w_x = torch.pow(w_x, power)
    scaling_factors_1 = w_x.unsqueeze(2).repeat(1, 1, k)
    multiplier1 = torch.gather(scaling_factors_1, 1, nn[1].idx)

    scaled_dist_1 = torch.mul(nn[1].dists, multiplier1)
    scaled_dist_1x, i1 = torch.min(scaled_dist_1, 2)

    # measure density with itself
    nn_y = chamferDist(y, y, bidirectional=False, return_nn=True, k=k2)
    density_y = torch.mean(nn_y[0].dists[:,:,1:], dim=2)
    density_y = 1 / (density_y + eps)
    high, low = torch.max(density_y), torch.min(density_y)
    diff = high - low + eps
    density_y = (density_y - low) / diff

    # measure density with other cloud
    density_yx = torch.mean(nn[1].dists[:,:,:k2-1], dim=2)
    density_yx = 1 / (density_yx + eps)
    high, low = torch.max(density_yx), torch.min(density_yx)
    diff = high - low + eps
    density_yx = (density_yx - low) / diff
    w_y = torch.div(density_yx, density_y)
    w_y = torch.pow(w_y, power)
    scaling_factors_0 = w_y.unsqueeze(2).repeat(1, 1, k)
    multiplier0 = torch.gather(scaling_factors_0, 1, nn[0].idx)

    scaled_dist_0 = torch.mul(nn[0].dists, multiplier0)
    scaled_dist_0x, i0 = torch.min(scaled_dist_0, 2)

    min_dist_1 = torch.gather(nn[1].dists, 2, i1.unsqueeze(2).repeat(1,1,k))[:, :, 0]
    min_dist_0 = torch.gather(nn[0].dists, 2, i0.unsqueeze(2).repeat(1,1,k))[:, :, 0]

    balanced_cd = torch.sum(torch.sqrt(min_dist_0)) + torch.sum(torch.sqrt(min_dist_1))

    batch_size, point_count, _ = x.shape
    bidirectional_dist = balanced_cd / (batch_size * point_count)

    if return_dists:
        return min_dist_0, min_dist_1

    if return_assignment:
        min_ind_1 = torch.gather(nn[1].idx, 2, i1.unsqueeze(2).repeat(1,1,k))[:, :, 0]
        min_ind_0 = torch.gather(nn[0].idx, 2, i0.unsqueeze(2).repeat(1,1,k))[:, :, 0]
        return bidirectional_dist, [min_ind_0, min_ind_1], [min_dist_0, min_dist_1] # return tensors, plus raw dists for outlier rejection
    else:
        return bidirectional_dist
    
    
    
    
def calc_dcd_correspondence_tensor(x, y, k=32, return_assignment=False, return_dists=False):

    chamferDist = ChamferDistance()
    nn = chamferDist(
        x,
        y,
        bidirectional=True,
        return_nn=True,
        k=k
    )

    eps = 0.00001
    batch_size, point_count, _ = x.shape

    # softmaxed_0 = torch.nn.functional.softmax(1/(nn[0].dists+eps), dim=-1)
    # softmaxed_1 = torch.nn.functional.softmax(1/(nn[1].dists+eps), dim=-1)
    softmaxed_0 = torch.nn.functional.softmin(nn[0].dists, dim=-1)
    softmaxed_1 = torch.nn.functional.softmin(nn[1].dists, dim=-1)

    point_weights_1 = torch.zeros(batch_size, point_count, dtype=softmaxed_0.dtype, device=softmaxed_0.device)
    for i in range(batch_size):
        point_weights_1[i].scatter_add_(0, nn[0].idx[i].flatten(), softmaxed_0[i].flatten())

    point_weights_0 = torch.zeros(batch_size, point_count, dtype=softmaxed_1.dtype, device=softmaxed_1.device)
    for i in range(batch_size):
        point_weights_0[i].scatter_add_(0, nn[1].idx[i].flatten(), softmaxed_1[i].flatten())

    # Use advanced indexing to gather point weights and calculate weighted distances
    corresponding_weights_0 = point_weights_1.unsqueeze(2).repeat(1,1,k).gather(1, nn[0].idx)
    corresponding_weights_1 = point_weights_0.unsqueeze(2).repeat(1,1,k).gather(1, nn[1].idx)

    corresponding_weights_0 = torch.mul(nn[0].dists, corresponding_weights_0)
    corresponding_weights_1 = torch.mul(nn[1].dists, corresponding_weights_1)

    _, i0 = torch.min(corresponding_weights_0, dim=2)
    _, i1 = torch.min(corresponding_weights_1, dim=2)

    min_dist_1 = torch.gather(nn[1].dists, 2, i1.unsqueeze(2).repeat(1,1,k))[:, :, 0]
    min_dist_0 = torch.gather(nn[0].dists, 2, i0.unsqueeze(2).repeat(1,1,k))[:, :, 0]

    # infoCD modification
    dist1 = torch.clamp(min_dist_0, min=1e-9)
    dist2 = torch.clamp(min_dist_1, min=1e-9)
    d1 = torch.sqrt(dist1)
    d2 = torch.sqrt(dist2)

    distances1 = - torch.log(torch.exp(-0.5 * d1)/(torch.sum(torch.exp(-0.5 * d1) + 1e-7,dim=-1).unsqueeze(-1))**1e-7)
    distances2 = - torch.log(torch.exp(-0.5 * d2)/(torch.sum(torch.exp(-0.5 * d2) + 1e-7,dim=-1).unsqueeze(-1))**1e-7)

    infocd =  (torch.sum(distances1) + torch.sum(distances2)) / 2
    infocd = infocd / (batch_size * point_count)

    # return infocd

    dcd = torch.sum(torch.sqrt(min_dist_1)) + torch.sum(torch.sqrt(min_dist_0))
    dcd = dcd / (batch_size * point_count)

    # corresponding_weights_1 = point_weights_0.gather(1, nn[1].idx)

    #print("corres", corresponding_weights_0.shape, i0.shape, min_dist_0.shape)


    # print("dcd", dcd.item(), bidirectional_dist.item())

    if return_assignment:
        min_ind_1 = torch.gather(nn[1].idx, 2, i1.unsqueeze(2).repeat(1,1,k))[:, :, 0]
        min_ind_0 = torch.gather(nn[0].idx, 2, i0.unsqueeze(2).repeat(1,1,k))[:, :, 0]

        return dcd, [min_ind_0, min_ind_1], [min_dist_0, min_dist_1]

    return dcd

In [ ]:
def draw_registration_result(source, target, transformation):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    source_temp.paint_uniform_color([1, 0.706, 0])
    target_temp.paint_uniform_color([0, 0.651, 0.929])
    source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp],
                                      zoom=0.4459,
                                      front=[0.9288, -0.2951, -0.2242],
                                      lookat=[1.6784, 2.0612, 1.4451],
                                      up=[-0.3402, -0.9189, -0.1996])

In [ ]:
# load and subsample

demo_icp_pcds = o3d.data.DemoICPPointClouds().paths

target_cloud = o3d.io.read_point_cloud(demo_icp_pcds[1])

trans_init = np.asarray([[0.862, 0.011, -0.507, 0.5],
                         [-0.139, 0.967, -0.215, 0.7],
                         [0.487, 0.255, 0.835, -1.4], [0.0, 0.0, 0.0, 1.0]])
source_cloud = o3d.io.read_point_cloud(demo_icp_pcds[0]).transform(trans_init)

target_cloud_down = target_cloud.voxel_down_sample(voxel_size=0.02)
source_cloud_down = source_cloud.voxel_down_sample(voxel_size=0.02)

# Convert to GPU tensors of shape (1, N, 3) 
cuda = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
source_pts = torch.tensor(np.array(source_cloud_down.points), dtype=torch.float32, device=cuda).unsqueeze(0)
target_pts = torch.tensor(np.array(target_cloud_down.points), dtype=torch.float32, device=cuda).unsqueeze(0)

In [ ]:

# named tuples for inputs/outputs
class SimilarityTransform(NamedTuple):
    R: torch.Tensor
    T: torch.Tensor
    s: torch.Tensor

class ICPSolution(NamedTuple):
    converged: bool
    rmse: Union[torch.Tensor, None]
    Xt: torch.Tensor
    RTs: SimilarityTransform
    t_history: List[SimilarityTransform]

def iterative_closest_point(
    X: Union[torch.Tensor, "Pointclouds"],
    Y: Union[torch.Tensor, "Pointclouds"],
    init_transform: Optional[SimilarityTransform] = None,
    max_iterations: int = 100,
    relative_rmse_thr: float = 1e-6,
    estimate_scale: bool = False,
    allow_reflection: bool = False,
    verbose: bool = False,
    correspondence_metric: str = "balanced_chamfer",
) -> ICPSolution:
    """
    correspondence_metric: str
        Either "knn" for standard nearest neighbor, "balanced_chamfer" for density-weighted matching,
        or "dcd" for density-aware Chamfer with information-theoretic weighting
    """
    
    Xt, num_points_X = oputil.convert_pointclouds_to_tensor(X)
    Yt, num_points_Y = oputil.convert_pointclouds_to_tensor(Y)

    b, size_X, dim = Xt.shape

    if (Xt.shape[2] != Yt.shape[2]) or (Xt.shape[0] != Yt.shape[0]):
        raise ValueError("Point sets X and Y have to have the same number of batches and data dimensions.")

    if ((num_points_Y < Yt.shape[1]).any() or (num_points_X < Xt.shape[1]).any()) and (num_points_Y != num_points_X).any():
        mask_X = (torch.arange(size_X, dtype=torch.int64, device=Xt.device)[None] < num_points_X[:, None]).type_as(Xt)
    else:
        mask_X = Xt.new_ones(b, size_X)

    Xt_init = Xt.clone()

    if init_transform is not None:
        R, T, s = init_transform
        Xt = _apply_similarity_transform(Xt, R, T, s)
    else:
        R = oputil.eyes(dim, b, device=Xt.device, dtype=Xt.dtype)
        T = Xt.new_zeros((b, dim))
        s = Xt.new_ones(b)

    prev_rmse = None
    rmse = None
    iteration = -1
    converged = False
    t_history = []

    # We need a distance threshold for outlier rejection (same as what Open3D uses)
    dist_threshold = 0.2
    
    for iteration in range(max_iterations):
        # -----------------------------------------------------------------------------------
        # EXPERIMENT AREA: Swap between correspondence metrics
        # -----------------------------------------------------------------------------------
        
        if correspondence_metric == "knn":
            # Standard KNN nearest neighbor matching
            knn_res = knn_points(
                Xt, Yt, lengths1=num_points_X, lengths2=num_points_Y, K=1, return_nn=True
            )
            Xt_nn_points = knn_res.knn[:, :, 0, :]
            mapped_Y_dists = knn_res.dists[:, :, 0]  # knn_points returns squared distances
            
        elif correspondence_metric == "balanced_chamfer":
            # Custom density-weighted Chamfer matching
            dist, indices, custom_dists = calc_balanced_chamfer_loss_tensor(Xt, Yt, k=32, return_assignment=True)
            best_Y_indices = indices[0]
            mapped_Y_dists = custom_dists[0]  # To perform thresholding
            Xt_nn_points = torch.gather(Yt, 1, best_Y_indices.unsqueeze(-1).expand(-1, -1, dim))
            
        elif correspondence_metric == "dcd":
            # Density-aware Chamfer with information-theoretic weighting
            dist, indices, custom_dists = calc_dcd_correspondence_tensor(Xt, Yt, k=32, return_assignment=True)
            best_Y_indices = indices[0]
            mapped_Y_dists = custom_dists[0]  # To perform thresholding
            Xt_nn_points = torch.gather(Yt, 1, best_Y_indices.unsqueeze(-1).expand(-1, -1, dim))
            
        else:
            raise ValueError(f"Unknown correspondence_metric: {correspondence_metric}. Use 'knn', 'balanced_chamfer', or 'dcd'")
        
        # OUTLIER REJECTION: Mask out point pairs that are farther apart than our threshold
        valid_pairs_mask = (mapped_Y_dists < (dist_threshold ** 2)).type_as(mask_X)
        current_weights = mask_X * valid_pairs_mask

        R, T, s = corresponding_points_alignment(
            Xt_init,
            Xt_nn_points,
            weights=current_weights,
            estimate_scale=estimate_scale,
            allow_reflection=allow_reflection,
        )

        Xt = _apply_similarity_transform(Xt_init, R, T, s)
        t_history.append(SimilarityTransform(R, T, s))

        # compute the root mean squared error ONLY on the valid masked pairs
        Xt_sq_diff = torch.square(Xt - Xt_nn_points).sum(2)
        # Avoid zero-division if no valid pairs
        if current_weights.sum() > 0:
            rmse = oputil.wmean(Xt_sq_diff[:, :, None], current_weights).sqrt()[:, 0, 0]
        else:
            rmse = Xt_sq_diff.new_ones(b) * float('inf')

        if prev_rmse is None:
            relative_rmse = rmse.new_ones(b)
        else:
            relative_rmse = (prev_rmse - rmse) / prev_rmse

        if verbose:
            print(f"ICP iteration {iteration}: mean/max rmse = {rmse.mean():1.2e}/{rmse.max():1.2e} ; mean relative rmse = {relative_rmse.mean():1.2e}")

        if (relative_rmse <= relative_rmse_thr).all():
            converged = True
            break

        prev_rmse = rmse

    if verbose:
        print(f"ICP has {'converged' if converged else 'not converged'} in {iteration + 1} iterations.")

    if hasattr(X, "update_padded"): # pointclouds object
        Xt = X.update_padded(Xt)

    return ICPSolution(converged, rmse, Xt, SimilarityTransform(R, T, s), t_history)


In [ ]:
AMBIGUOUS_ROT_SINGULAR_THR = 1e-15

def corresponding_points_alignment(
    X: Union[torch.Tensor, "Pointclouds"],
    Y: Union[torch.Tensor, "Pointclouds"],
    weights: Union[torch.Tensor, List[torch.Tensor], None] = None,
    estimate_scale: bool = False,
    allow_reflection: bool = False,
    eps: float = 1e-9,
) -> SimilarityTransform:

    Xt, num_points = oputil.convert_pointclouds_to_tensor(X)
    Yt, num_points_Y = oputil.convert_pointclouds_to_tensor(Y)

    if (Xt.shape != Yt.shape) or (num_points != num_points_Y).any():
        raise ValueError("Point sets X and Y have to have the same number of batches, points and dimensions.")

    if weights is not None:
        if isinstance(weights, list):
            weights = [w[..., None] for w in weights]
            weights = strutil.list_to_padded(weights)[..., 0]

    b, n, dim = Xt.shape

    Xmu = oputil.wmean(Xt, weight=weights, eps=eps)
    Ymu = oputil.wmean(Yt, weight=weights, eps=eps)

    Xc = Xt - Xmu
    Yc = Yt - Ymu

    total_weight = torch.clamp(num_points, 1)
    if weights is not None:
        Xc *= weights[:, :, None]
        Yc *= weights[:, :, None]
        total_weight = torch.clamp(weights.sum(1), eps)

    XYcov = torch.bmm(Xc.transpose(2, 1), Yc) / total_weight[:, None, None]
    U, S, V = torch.svd(XYcov)

    E = torch.eye(dim, dtype=XYcov.dtype, device=XYcov.device)[None].repeat(b, 1, 1)

    if not allow_reflection:
        R_test = torch.bmm(U, V.transpose(2, 1))
        E[:, -1, -1] = torch.det(R_test)

    R = torch.bmm(torch.bmm(U, E), V.transpose(2, 1))

    if estimate_scale:
        trace_ES = (torch.diagonal(E, dim1=1, dim2=2) * S).sum(1)
        Xcov = (Xc * Xc).sum((1, 2)) / total_weight
        s = trace_ES / torch.clamp(Xcov, eps)
        T = Ymu[:, 0, :] - s[:, None] * torch.bmm(Xmu, R)[:, 0, :]
    else:
        T = Ymu[:, 0, :] - torch.bmm(Xmu, R)[:, 0, :]
        s = T.new_ones(b)

    return SimilarityTransform(R, T, s)

def _apply_similarity_transform(
    X: torch.Tensor, R: torch.Tensor, T: torch.Tensor, s: torch.Tensor
) -> torch.Tensor:
    X = s[:, None, None] * torch.bmm(X, R) + T[:, None, :]
    return X

In [ ]:
# Run the baseline PyTorch3D ICP
print("Starting PyTorch3D ICP...")

# Give it an identity transform as starting point (or trans_init if you prefer)
init_R = torch.eye(3, device=cuda).unsqueeze(0)
init_T = torch.zeros((1, 3), device=cuda)
init_s = torch.ones(1, device=cuda)

solution = iterative_closest_point(
    X=source_pts,
    Y=target_pts,
    init_transform=SimilarityTransform(init_R, init_T, init_s),
    max_iterations=50,
    verbose=True
)

print(f"Converged: {solution.converged}, Final RMSE: {solution.rmse.item():.5f}")

In [ ]:
# Compare all three metrics
print("=" * 80)
print("COMPARING CORRESPONDENCE METRICS")
print("=" * 80)

# Test with KNN (standard nearest neighbor)
print("\n1. Running ICP with KNN (standard nearest neighbor)...")
init_R = torch.eye(3, device=cuda).unsqueeze(0)
init_T = torch.zeros((1, 3), device=cuda)
init_s = torch.ones(1, device=cuda)

solution_knn = iterative_closest_point(
    X=source_pts,
    Y=target_pts,
    init_transform=SimilarityTransform(init_R, init_T, init_s),
    max_iterations=50,
    verbose=False,
    correspondence_metric="knn"
)
loss_knn = ChamferDistance()(target_pts, solution_knn.Xt, bidirectional=True).item()
print(f"   Converged: {solution_knn.converged} | Final RMSE: {solution_knn.rmse.item():.6f} | Chamfer Distance: {loss_knn:.6f}")

# Test with balanced chamfer loss
print("\n2. Running ICP with Balanced Chamfer Loss (density-weighted)...")
solution_balanced = iterative_closest_point(
    X=source_pts,
    Y=target_pts,
    init_transform=SimilarityTransform(init_R, init_T, init_s),
    max_iterations=50,
    verbose=False,
    correspondence_metric="balanced_chamfer"
)
loss_balanced = ChamferDistance()(target_pts, solution_balanced.Xt, bidirectional=True).item()
print(f"   Converged: {solution_balanced.converged} | Final RMSE: {solution_balanced.rmse.item():.6f} | Chamfer Distance: {loss_balanced:.6f}")

# Test with DCD (Density-aware Chamfer Distance)
print("\n3. Running ICP with DCD (Density-Aware Chamfer Distance)...")
solution_dcd = iterative_closest_point(
    X=source_pts,
    Y=target_pts,
    init_transform=SimilarityTransform(init_R, init_T, init_s),
    max_iterations=50,
    verbose=False,
    correspondence_metric="dcd"
)
loss_dcd = ChamferDistance()(target_pts, solution_dcd.Xt, bidirectional=True).item()
print(f"   Converged: {solution_dcd.converged} | Final RMSE: {solution_dcd.rmse.item():.6f} | Chamfer Distance: {loss_dcd:.6f}")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
results = {
    "KNN": loss_knn,
    "Balanced Chamfer": loss_balanced,
    "DCD": loss_dcd
}
best_metric = min(results, key=results.get)
best_loss = results[best_metric]

for metric, loss in results.items():
    marker = "✓ " if metric == best_metric else "  "
    print(f"{marker}{metric:20s}: {loss:.6f}")

print("\n" + "=" * 80)
print(f"Best Performance: {best_metric} ({best_loss:.6f})")
print("=" * 80)


In [ ]:
# Compute Chamfer Distance on the result
chamferDist = ChamferDistance()

# Check final bidirectional chamfer loss
loss = chamferDist(target_pts, solution.Xt, bidirectional=True)
print(f"Bidirectional Chamfer Distance after PyTorch3D ICP: {loss.item()}")


In [ ]:
# Visualise the result using Open3D

R = solution.RTs.R[0].cpu().numpy()
T = solution.RTs.T[0].cpu().numpy()
s = solution.RTs.s[0].cpu().numpy()

# Note: PyTorch3d calculates Right-Multiplied Matrices ( X_new = s * X @ R + T)
# Open3D calculates Left-Multiplied Matrices ( X_new = s * R @ X + T )
# Therefore, we have to Transpose the Rotation matrix for Open3D (R.T)

transform_matrix = np.eye(4)
transform_matrix[:3, :3] = R.T * s
transform_matrix[:3, 3] = T

print("Final 4x4 Transformation Matrix:")
print(transform_matrix)

# Visualize!
draw_registration_result(source_cloud_down, target_cloud_down, transform_matrix)